In [1]:
import pathlib, sys
import matplotlib
matplotlib.use('Agg')
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# 02 · What Cleaning Destroys

Abstract claims about 'information loss' become concrete when you watch cleaning change the **meaning** of a real sentence. This notebook shows the four kinds of damage: deleted negations, collapsed casing, erased punctuation, and stemming into non-words.

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from src.cleaning.steps import lowercase, remove_stopwords, stem, lemmatize, strip_punctuation
from src.cleaning.pipelines import PIPELINE_NAMES, clean
from src.cleaning.analysis import pipeline_table
from src.datasets.corpora import load_adversarial, load_sentiment
from src.evaluation.failure_cases import negations_removed
from src.visualisation import save_figure, save_json
pd.set_option('display.max_colwidth', 70)

## The adversarial showcase

Hand-built sentences where a single cleaning step flips or guts the meaning.

In [3]:
adv = load_adversarial()
adv_df = pd.DataFrame([{
    'phenomenon': ex.phenomenon, 'original': ex.text,
    'standard': clean(ex.text, 'standard'), 'aggressive': clean(ex.text, 'aggressive'),
    'deleted_negations': ', '.join(sorted(negations_removed(ex.text, 'standard'))),
    'note': ex.note,
} for ex in adv])
save_json(adv_df, 'nb02_adversarial.json')
adv_df

,phenomenon,original,standard,aggressive,deleted_negations,note
0,negation,The movie was not good.,movie good.,movi good,not,"Stopword removal deletes 'not', turning a negative into 'good'."
1,negation,I would never recommend this.,would recommend,would recommend,never,"Dropping 'never' leaves 'would recommend', flipping the label."
2,negation,This is no masterpiece.,masterpiece.,masterpiec,no,Removing 'no' leaves the praise word 'masterpiece' standing alone.
3,casing,Apple fell as the US dollar rose.,apple fell dollar rose.,appl fell dollar rose,,Lowercasing collapses 'Apple'->'apple' and 'US'->'us'.
4,punctuation,"Let's eat, Grandma.","let's eat, grandma.",let eat grandma,,Stripping the comma erases the difference from 'Let's eat Grandma'.
5,emoticon,The studies were inconclusive :(,study inconclusive :(,studi inconclus,,Punctuation stripping deletes the ':(' sentiment cue.


## 1. Negation deletion — the label flip

Stopword removal deletes `not` / `no` / `never`. `not good` becomes `good`.

In [4]:
neg_examples = [
    'not good',
    'I would never recommend this',
    'there is no better film this year',
    'it was not a waste of time',
]
neg_df = pd.DataFrame([{
    'original': s, 'remove_stopwords': remove_stopwords(s), 'standard': clean(s, 'standard'),
    'deleted': ', '.join(sorted(negations_removed(s, 'standard'))),
} for s in neg_examples])
save_json(neg_df, 'nb02_negation_examples.json')
neg_df

,original,remove_stopwords,standard,deleted
0,not good,good,good,not
1,I would never recommend this,would recommend,would recommend,never
2,there is no better film this year,better film year,better film year,no
3,it was not a waste of time,waste time,waste time,not


In [5]:
texts, labels = load_sentiment()
rep_map = {r.name: r.negations_removed for r in pipeline_table(texts)}
dmg_df = pd.DataFrame([{
    'pipeline': p,
    'docs_damaged': sum(1 for t in texts if negations_removed(t, p)),
    'negations_removed_total': rep_map[p],
} for p in PIPELINE_NAMES])
save_json(dmg_df, 'nb02_negations_removed_by_pipeline.json')
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(dmg_df['pipeline'], dmg_df['negations_removed_total'], color='#c44e52')
ax.set_ylabel('negation words deleted'); ax.set_xlabel('pipeline')
ax.set_title('Negations deleted from the sentiment corpus by each pipeline')
fig.tight_layout()
save_figure(fig, 'nb02_negations_removed_by_pipeline.png')
dmg_df

,pipeline,docs_damaged,negations_removed_total
0,none,0,0
1,minimal,0,0
2,standard,33,33
3,aggressive,33,33


## 2. Casing collapse

Lowercasing merges distinct tokens: the company `Apple` and the fruit `apple`, the country `US` and the pronoun `us`.

In [6]:
casing_pairs = [('US', 'us'), ('Apple', 'apple'), ('March', 'march'), ('Polish', 'polish')]
casing_df = pd.DataFrame([{
    'token_a': a, 'token_b': b, 'lowercased': lowercase(a), 'collapsed': lowercase(a) == lowercase(b),
} for a, b in casing_pairs])
save_json(casing_df, 'nb02_casing_collapse.json')
casing_df

,token_a,token_b,lowercased,collapsed
0,US,us,us,True
1,Apple,apple,apple,True
2,March,march,march,True
3,Polish,polish,polish,True


## 3. Punctuation removal

Stripping punctuation erases the structure that separates two very different sentences.

In [7]:
punct = ["Let's eat, grandma!", "Let's eat grandma!", 'the price is $9.99, not $99']
punct_df = pd.DataFrame([{'original': s, 'strip_punctuation': strip_punctuation(s)} for s in punct])
save_json(punct_df, 'nb02_punctuation.json')
punct_df

,original,strip_punctuation
0,"Let's eat, grandma!",Let s eat grandma
1,Let's eat grandma!,Let s eat grandma
2,"the price is $9.99, not $99",the price is 9 99 not 99


## 4. Stemming into non-words

Aggressive stemming over-chops into forms a transformer's tokenizer never saw (`studi`, `happi`), while lemmatization returns valid words.

In [8]:
words = ['studies', 'running', 'happiness', 'universities', 'organization', 'flies', 'better']
stem_df = pd.DataFrame([{
    'word': w, 'stem': stem(w), 'lemmatize': lemmatize(w), 'differ': stem(w) != lemmatize(w),
} for w in words])
save_json(stem_df, 'nb02_stemming_vs_lemmatize.json')
stem_df

,word,stem,lemmatize,differ
0,studies,studi,study,True
1,running,run,running,True
2,happiness,happi,happiness,True
3,universities,univers,university,True
4,organization,organ,organization,True
5,flies,fli,fly,True
6,better,better,better,False


## Takeaway

Every one of these is irreversible: once the text is lowercased, de-punctuated, de-negated, and stemmed, no downstream model can recover what was there. For a bag-of-words model that lost information was mostly noise. For a transformer pretrained on raw text, it was signal. The next notebook measures the gap.